# Phase 6 Dual-Path : FINAL_VALIDATION (BS30) + 3-way comparison

Replicates the **BS30 FINAL_VALIDATION** protocol from
`st_cdgm_noncausal_training.ipynb` Cell 61 on the **phase6_dualpath** Stage 1
stack (encoder + RCN + head + dual_path), then writes JSONs in
`/content/drive/MyDrive/climate_data/ckpt_phase6_dualpath/` whose schema is
**identical** to the causal and non-causal references. The 3-way comparison cell
loads all three reference dirs and produces a winners table consumable by
`st_cdgm_causal_vs_noncausal_comparison.ipynb`.

**Protocol** (BS30, hard-coded in Cell 2):
- `N_TEST_BATCHES = 16`, `K_SAMPLES = 64`, `N_STEPS_DIFF = 18`
- `scheduler = edm_karras`, `cfg_scale = 0.0`, `causal_concat = True`
- mu_HR ablation on first `N_INTERVENTION = 4` batches
- `sigma_data = SIGMA_DATA_NEW = 0.193` (Phase 6 mu_total regime)

**Outputs in `PHASE6_REF_DIR`:**
- `final_validation_metrics.json`  (BS30, Cell 6)
- `eval_samples.npz`               (BS43-style, Cell 6)
- `domain_metrics.json`            (BS42, Cell 7)
- `aligned_metrics_<gcm>_dualpath.json`  (BS44, Cell 8)
- `comparison_3way.json`           (Cell 9)


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab - Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Constants for BS30 FINAL_VALIDATION protocol on phase6_dualpath ===
import json
import numpy as np
import torch
from pathlib import Path
from omegaconf import OmegaConf

from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last

# --- Drive paths ---
DRIVE_ROOT     = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N      = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_DUALPATH  = ORACLE_9N / 'epoch_best_dualpath.pth'
CKPT_STAGE2    = ORACLE_9N / 'epoch_last.pth'
SIGMA_DATA_NEW = 0.193

# Existing reference JSONs to compare against (allow override via globals()).
CAUSAL_REF_DIR    = Path(str(globals().get('CAUSAL_REF_DIR',
    DRIVE_ROOT / 'ckpt_v2_corrdiff_normal')))
NONCAUSAL_REF_DIR = Path(str(globals().get('NONCAUSAL_REF_DIR',
    DRIVE_ROOT / 'ckpt_noncausal')))

# Output dir for phase6_dualpath validation results (= a NEW reference folder).
PHASE6_REF_DIR    = Path(str(globals().get('PHASE6_REF_DIR',
    DRIVE_ROOT / 'ckpt_phase6_dualpath')))
PHASE6_REF_DIR.mkdir(parents=True, exist_ok=True)

# BS30 protocol (EXACT same as non-causal training Cell 61).
N_TEST_BATCHES  = 16
K_SAMPLES       = 64
N_STEPS_DIFF    = 18
N_INTERVENTION  = 4

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cell 2] DEVICE          = {DEVICE}')
print(f'[Cell 2] CKPT_DUALPATH   = {CKPT_DUALPATH}  exists={CKPT_DUALPATH.exists()}')
print(f'[Cell 2] CKPT_STAGE2     = {CKPT_STAGE2}  exists={CKPT_STAGE2.exists()}')
print(f'[Cell 2] SIGMA_DATA_NEW  = {SIGMA_DATA_NEW}')
print(f'[Cell 2] CAUSAL_REF_DIR    = {CAUSAL_REF_DIR}')
print(f'[Cell 2] NONCAUSAL_REF_DIR = {NONCAUSAL_REF_DIR}')
print(f'[Cell 2] PHASE6_REF_DIR    = {PHASE6_REF_DIR}')
print(f'[Cell 2] BS30 protocol : N_TEST_BATCHES={N_TEST_BATCHES} K_SAMPLES={K_SAMPLES} '
      f'N_STEPS_DIFF={N_STEPS_DIFF} N_INTERVENTION={N_INTERVENTION}')
_total_diff_calls = N_TEST_BATCHES * K_SAMPLES * N_STEPS_DIFF + N_INTERVENTION * 2 * N_STEPS_DIFF
print(f'[Cell 2] Total diffusion calls = {_total_diff_calls:,} '
      f'(BS30 ensemble + mu_HR ablation)')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Forcer batch_size=1 pour la compatibilite single-sample (IterableDataset)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# --- Ajout metapaths 9-node ---
OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- Dates ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

# --- convert_sample_to_batch (identique training, avec lr_grid pour batch_lr_grid_last) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

# Constantes pour HR shape utilises plus tard pour DualPathPredictor
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Load Stage 2 (diffusion decoder) ===
# Detects num_vars from the Phase 6 dualpath ckpt encoder metapaths, then loads
# the shared diffusion_decoder with sigma_data = SIGMA_DATA_NEW = 0.193.
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

print(f'[Cell 4] Loading Stage 2 : {CKPT_STAGE2}')
ck_s2 = torch.load(CKPT_STAGE2, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] keys[:12] = {sorted(ck_s2.keys())[:12]}')

# Detect num_vars from Phase 6 dualpath encoder metapaths.
_ck_peek = torch.load(CKPT_DUALPATH, map_location='cpu', weights_only=False)
_enc_sd_peek = _ck_peek.get('encoder_state_dict', {})
_metapath_names = set()
for k in _enc_sd_peek:
    if k.startswith('metapath_convs.'):
        _metapath_names.add(k[len('metapath_convs.'):].split('__')[0])
num_vars = len(_metapath_names)
del _ck_peek, _enc_sd_peek
print(f'[Cell 4] num_vars detected = {num_vars}')

# Build UNET_KWARGS with projection_class_embeddings_input_dim override.
UNET_KWARGS = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KWARGS and isinstance(UNET_KWARGS[_k], list):
        UNET_KWARGS[_k] = tuple(UNET_KWARGS[_k])
UNET_KWARGS['projection_class_embeddings_input_dim'] = (
    num_vars * int(CONFIG.diffusion.conditioning_dim))
print(f'[Cell 4] projection_class_embeddings_input_dim = '
      f'{UNET_KWARGS["projection_class_embeddings_input_dim"]}')

edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])

diffusion_decoder = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height),
    width=int(CONFIG.diffusion.width),
    unet_kwargs=UNET_KWARGS,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=bool(CONFIG.diffusion.get('use_gradient_checkpointing', False)),
    conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
    edm_config=edm_cfg,
    causal_concat=True,
).to(DEVICE)


def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out


_diff_sd = _strip_prefixes(ck_s2.get('diffusion_state_dict'))
if _diff_sd is None:
    raise RuntimeError(
        f'diffusion_state_dict absent from {CKPT_STAGE2}. Stage 2 ckpt must be '
        f'the epoch_last.pth from Path C+ Option C 9-node training.'
    )
_missing, _unexpected = diffusion_decoder.load_state_dict(_diff_sd, strict=False)
if _missing:
    print(f'[Cell 4] missing keys : {len(_missing)} (first 3: {_missing[:3]})')
if _unexpected:
    print(f'[Cell 4] unexpected keys : {len(_unexpected)} (first 3: {_unexpected[:3]})')

# Override sigma_data for phase6_dualpath regime (mu_total).
_SIGMA_DATA_CKPT = float(diffusion_decoder.edm_config.sigma_data)
diffusion_decoder.edm_config.sigma_data = float(SIGMA_DATA_NEW)
print(f'[Cell 4] sigma_data : ckpt={_SIGMA_DATA_CKPT:.5f} -> new={SIGMA_DATA_NEW:.5f}')

for p in diffusion_decoder.parameters():
    p.requires_grad_(False)
diffusion_decoder.eval()
print(f'[Cell 4] Stage 2 ready (params={sum(p.numel() for p in diffusion_decoder.parameters()):,})')


In [ ]:
# === Cell 5 : Load Stage 1 dual-path (encoder + RCN + head + dual_path) ===
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder


def _parse_encoder_metapaths_from_ckpt(enc_sd):
    seen, order = {}, []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]; src = parts[1]; rel = parts[2]; tgt = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt); order.append(name)
    return [(n,) + seen[n] for n in order]


def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd


def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] loaded from "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] FAILED with "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] no valid key found in {keys}')
    return False


print(f'[Cell 5] Loading Stage 1 dual-path : {CKPT_DUALPATH}')
ck_s1 = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
print(f'[Cell 5] ckpt keys[:12] = {sorted(ck_s1.keys())[:12]}')

enc_sd = _clean_sd(ck_s1.get('encoder_state_dict', {}))
parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
print(f'  metapaths detected : {[t[0] for t in parsed]}')
cfgs = [
    IntelligibleVariableConfig(name=n, meta_path=(s, r, t), pool='mean')
    for n, s, r, t in parsed
]
encoder = IntelligibleVariableEncoder(
    configs=cfgs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
n_vars = len(cfgs)

_probe_b = next(iter(val_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=n_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
GATE_MAX_MEAN        = 0.40
dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

_safe_load(encoder,         ck_s1, ['encoder_state_dict'],                            'encoder')
_safe_load(rcn_cell,        ck_s1, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
_safe_load(regression_head, ck_s1, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path,       ck_s1, ['dual_path_state_dict'],                          'dual_path')

for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        p.requires_grad_(False)
    m.eval()

_rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
if hasattr(_rcn_core, 'A_dag'):
    _A_dag = _rcn_core.A_dag.detach()
    print(f'  A_dag shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
          f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')

_n_enc = sum(p.numel() for p in encoder.parameters())
_n_rcn = sum(p.numel() for p in rcn_cell.parameters())
_n_rh  = sum(p.numel() for p in regression_head.parameters())
_n_dp  = sum(p.numel() for p in dual_path.parameters())
print(f'  param counts : enc={_n_enc:,}  rcn={_n_rcn:,}  head={_n_rh:,}  dual_path={_n_dp:,}')
print(f'  path_b_bias  : {float(dual_path.path_b_bias.item()):+.5f}')
print(f'[Cell 5] Stage 1 dual-path ready.')


In [ ]:
# === Cell 6 : BS30 FINAL_VALIDATION protocol on phase6_dualpath ===
# Mirrors st_cdgm_noncausal_training.ipynb Cell 61 line-by-line, swapping the
# Stage 1 path for the Phase 6 dual-path stack (encoder + RCN + head + dual_path).
# Saves final_validation_metrics.json + eval_samples.npz in PHASE6_REF_DIR,
# format compatible with st_cdgm_causal_vs_noncausal_comparison.ipynb.
import json
import time
import numpy as np
import torch

from st_cdgm.evaluation import compute_f1_extremes, compute_spectrum_distance


def _build_inputs(_batch):
    """Returns (conditioning=None, mu_total, baseline_log, target) for causal_concat."""
    # Path A : causal forward (encoder + RCN + head).
    lr_data = _batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(_batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A    = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3:
        mu_A = mu_A.unsqueeze(0)

    # Dual-path fusion : mu_total = mu_A + gate * mu_B(LR).
    lr_grid = batch_lr_grid_last(_batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, _mu_B, _gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    # baseline_log : last step of the input sequence.
    bl = _batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    # target : HR residual at the last step.
    tgt = _batch['residual'][-1].to(DEVICE)
    if tgt.dim() == 3:
        tgt = tgt.unsqueeze(0)

    # cond = None : causal_concat uses mu_HR + baseline_log directly.
    return None, mu_total, bl, tgt


def _sample_once(_cond, _mu_HR, _baseline_log):
    out = diffusion_decoder.sample(
        conditioning=None,
        num_steps=N_STEPS_DIFF,
        scheduler_type='edm_karras',
        cfg_scale=0.0,
        apply_constraints=False,
        mu_HR=_mu_HR,
        baseline_log=_baseline_log,
    )
    return out.residual if hasattr(out, 'residual') else out


print(f'[Cell 6] Sampling {N_TEST_BATCHES} batches x {K_SAMPLES} samples '
      f'x {N_STEPS_DIFF} diff steps (scheduler=edm_karras, cfg=0.0)')

_all_means = []
_all_stds = []
_all_targets = []
_all_mu_HR = []
_intervention = []
_t_eval_start = time.time()
_count = 0

with torch.no_grad():
    for converted_batches in iterate_batches(val_dataloader, builder, DEVICE):
        for _batch in converted_batches:
            if _count >= N_TEST_BATCHES:
                break
            _cond, _mu_HR, _baseline_log, _target = _build_inputs(_batch)
            _samples_k = torch.stack(
                [_sample_once(_cond, _mu_HR, _baseline_log) for _ in range(K_SAMPLES)],
                dim=0,
            )
            _all_means.append(_samples_k.mean(dim=0))
            _all_stds.append(_samples_k.std(dim=0))
            _all_targets.append(_target)
            _all_mu_HR.append(_mu_HR.detach())

            if _count < N_INTERVENTION:
                _mu_zero = torch.zeros_like(_mu_HR)
                _s_real = _sample_once(_cond, _mu_HR,   _baseline_log)
                _s_zero = _sample_once(_cond, _mu_zero, _baseline_log)
                _delta  = (_s_real - _s_zero).abs().mean().item()
                _signal = _s_real.abs().mean().item()
                _ratio  = _delta / max(_signal, 1e-8)
                _intervention.append(_ratio)
                print(f'   batch {_count+1:2d} | mu_HR ablation delta/signal = {_ratio*100:6.2f}%')
            else:
                print(f'   batch {_count+1:2d} | sampled K={K_SAMPLES}')

            _count += 1
        if _count >= N_TEST_BATCHES:
            break

_eval_time = time.time() - _t_eval_start
print(f'\n[Cell 6] Eval done in {_eval_time:.1f}s ({_eval_time/60:.1f} min)')

# === Aggregate ===
_pred_mean = torch.cat(_all_means,   dim=0).cpu()  # [N, 1, H, W] = delta_hat
_pred_std  = torch.cat(_all_stds,    dim=0).cpu()
_targets   = torch.cat(_all_targets, dim=0).cpu()
_mu_HR_cat = torch.cat(_all_mu_HR,   dim=0).cpu()

# In causal_concat mode the diffusion outputs delta_hat (residual on top of
# mu_HR). To measure end-to-end reconstruction skill, build the FULL prediction
# pred_full = mu_HR + delta_hat (BS31f_FULL_PREDICTION convention).
if _mu_HR_cat.shape == _pred_mean.shape:
    _pred_full = _mu_HR_cat + _pred_mean
    _metrics_scope = 'full_prediction (mu_HR + delta_hat)'
else:
    print(f'   [warn] mu_HR shape mismatch -> fallback raw delta_hat for metrics')
    _pred_full = _pred_mean
    _metrics_scope = 'delta_only (fallback)'
print(f'[Cell 6] metrics scope = {_metrics_scope}')

# === Metrics ===
_valid = torch.isfinite(_targets)

_pred_clean = torch.where(_valid, _pred_full, torch.zeros_like(_pred_full))
_targ_clean = torch.where(_valid, _targets,   torch.zeros_like(_targets))
_diff_sq = ((_pred_clean - _targ_clean) ** 2)[_valid]
_rmse  = float(_diff_sq.mean().sqrt().item()) if _valid.any() else float('nan')
_mae   = float((_pred_clean - _targ_clean).abs()[_valid].mean().item()) if _valid.any() else float('nan')
_spread = float(_pred_std[_valid].mean().item()) if _valid.any() else float('nan')


def _pearson(a, b, eps=1e-8):
    a_c = a - a.mean()
    b_c = b - b.mean()
    num = (a_c * b_c).sum()
    den = torch.sqrt((a_c * a_c).sum() * (b_c * b_c).sum() + eps)
    return float((num / den).item())


_corr_global = float('nan')
_corr_per_sample = float('nan')
_corr_per_sample_list = []
try:
    _p_flat = _pred_full[_valid]
    _t_flat = _targets[_valid]
    if _p_flat.numel() > 1 and _t_flat.numel() > 1:
        _corr_global = _pearson(_p_flat, _t_flat)
    for _i in range(_pred_full.shape[0]):
        _vi = _valid[_i]
        if _vi.sum() < 2:
            continue
        _c = _pearson(_pred_full[_i][_vi], _targets[_i][_vi])
        if _c == _c:
            _corr_per_sample_list.append(_c)
    if _corr_per_sample_list:
        _corr_per_sample = float(np.mean(_corr_per_sample_list))
except Exception as _e:
    print(f'[warn] Pearson failed : {_e}')

# F1 extremes : compute_f1_extremes returns {"p95": ..., "p99": ...}
_f1 = {}
try:
    _f1 = compute_f1_extremes(_pred_clean, _targ_clean, threshold_percentiles=[95.0, 99.0])
except Exception as _e:
    print(f'[warn] F1 extremes failed : {_e}')

# RAPSD distance on first sample (BS30 convention).
_rapsd_d = None
try:
    _rapsd_d = float(compute_spectrum_distance(_pred_clean[0], _targ_clean[0]))
except Exception as _e:
    print(f'[warn] RAPSD distance failed : {_e}')

# Shortcut diagnostic.
_shortcut = {
    'norm_output': None, 'norm_mu_HR': None, 'norm_target': None,
    'norm_output_minus_mu_HR': None, 'norm_target_minus_mu_HR': None,
    'shortcut_ratio': None, 'verdict': 'N/A',
}
if _mu_HR_cat.shape == _pred_mean.shape:
    _valid_mu = _valid & torch.isfinite(_mu_HR_cat)
    _out = _pred_mean[_valid_mu]
    _mu  = _mu_HR_cat[_valid_mu]
    _tg  = _targets[_valid_mu]
    _shortcut['norm_output']             = float(_out.abs().mean().item())
    _shortcut['norm_mu_HR']              = float(_mu.abs().mean().item())
    _shortcut['norm_target']             = float(_tg.abs().mean().item())
    _shortcut['norm_output_minus_mu_HR'] = float((_out - _mu).abs().mean().item())
    _shortcut['norm_target_minus_mu_HR'] = float((_tg - _mu).abs().mean().item())
    _ratio = (_shortcut['norm_output_minus_mu_HR']
              / max(_shortcut['norm_output'], 1e-12))
    _shortcut['shortcut_ratio'] = float(_ratio)
    if _ratio < 0.10:
        _shortcut['verdict'] = 'SHORTCUT_CONFIRMED'
    elif _ratio < 0.30:
        _shortcut['verdict'] = 'AMBIGUOUS'
    else:
        _shortcut['verdict'] = 'REFINEMENT_OK'

_dag_avg = float(np.mean(_intervention)) if _intervention else None
if _dag_avg is None:
    _verdict_dag = 'N/A'
elif _dag_avg < 0.001:
    _verdict_dag = 'MU_HR_IGNORED'
elif _dag_avg < 0.01:
    _verdict_dag = 'WEAK'
else:
    _verdict_dag = 'MU_HR_CONDITIONS'

# === Print ===
print('\n' + '=' * 72)
print('FINAL VALIDATION METRICS (BS30 protocol, phase6_dualpath)')
print('=' * 72)
print(f'  N batches             : {len(_all_targets)}')
print(f'  K samples / batch     : {K_SAMPLES}')
print(f'  Eval time             : {_eval_time:.1f}s  ({_eval_time/max(1,len(_all_targets)):.2f}s/batch)')
print()
print(f'  RMSE  (ensemble mean) : {_rmse:.6f}')
print(f'  MAE                   : {_mae:.6f}')
print(f'  Spread (ensemble std) : {_spread:.6f}')
print(f'  Pearson (global)      : {_corr_global:.4f}')
print(f'  Pearson (per-sample)  : {_corr_per_sample:.4f}  '
      f'(n={len(_corr_per_sample_list)})')
for _k, _v in _f1.items():
    print(f'  {_k:<22}: {_v:.4f}')
if _rapsd_d is not None:
    print(f'  RAPSD distance        : {_rapsd_d:.6f}')
if _shortcut['shortcut_ratio'] is not None:
    print(f'  Shortcut ratio        : {_shortcut["shortcut_ratio"]:.4f}  -> {_shortcut["verdict"]}')
if _dag_avg is not None:
    print(f'  mu_HR ablation        : {_dag_avg*100:.3f}%  -> {_verdict_dag}')

# === Save JSON ===
_metrics = {
    'checkpoint':       str(CKPT_DUALPATH),
    'mode':             'phase6_dualpath',
    'causal_concat':    True,
    'n_test_batches':   len(_all_targets),
    'k_samples':        K_SAMPLES,
    'metrics_scope':    _metrics_scope,
    'eval_time_s':      _eval_time,
    'rmse':             _rmse,
    'mae':              _mae,
    'spread_mean':      _spread,
    'f1_extremes':      _f1,
    'pearson_corr': {
        'global':          _corr_global,
        'per_sample_avg':  _corr_per_sample,
        'per_sample_n':    len(_corr_per_sample_list),
        'per_sample_list': _corr_per_sample_list[:64],
    },
    'rapsd_distance':   _rapsd_d,
    'mu_HR_ablation': {
        'delta_signal_ratio_avg': _dag_avg,
        'per_batch':              _intervention,
        'verdict':                _verdict_dag,
    },
    'shortcut_diagnostic': _shortcut,
    'eval_protocol': {
        'n_test_batches': N_TEST_BATCHES,
        'k_samples':      K_SAMPLES,
        'n_steps_diff':   N_STEPS_DIFF,
        'scheduler':      'edm_karras',
        'cfg_scale':      0.0,
    },
    'sigma_data': float(SIGMA_DATA_NEW),
}
_fv_path = PHASE6_REF_DIR / 'final_validation_metrics.json'
_fv_path.write_text(json.dumps(_metrics, indent=2, default=str), encoding='utf-8')
print(f'\n[Cell 6] Saved : {_fv_path}')

# === Save eval_samples.npz (BS43 format) ===
_N_dump = min(8, int(_targets.shape[0]))
_payload = dict(
    target    = _targets[:_N_dump].numpy(),
    pred_full = _pred_full[:_N_dump].numpy(),
    pred_std  = _pred_std[:_N_dump].numpy(),
    valid_mask= _valid[:_N_dump].numpy().astype('float32'),
    mu_HR     = _mu_HR_cat[:_N_dump].numpy(),
    run_variant = np.array('dualpath'),
)
_npz_path = PHASE6_REF_DIR / 'eval_samples.npz'
np.savez_compressed(_npz_path, **_payload)
print(f'[Cell 6] Saved : {_npz_path}  (N={_N_dump})')


In [ ]:
# === Cell 7 : domain_metrics.json (CorrDiff/Rampal style) ===
# Reuses _pred_full, _pred_std, _targets, _valid, _rmse, _spread, _mae, _corr_global
# from Cell 6. Saves to PHASE6_REF_DIR/domain_metrics.json (same format as BS42).
import math as _m42
import json as _j42

print('=' * 72)
print('DOMAIN-ALIGNED METRICS (phase6_dualpath, BS42 protocol)')
print('=' * 72)

_dm = {}

# 1. Spread-skill ratio (target ~1.0).
try:
    _ssr = float(_spread) / float(_rmse) if (_rmse == _rmse and _rmse > 0) else float('nan')
except Exception:
    _ssr = float('nan')
_dm['spread_skill_ratio'] = _ssr

# 2. CRPS gaussian closed-form.
_crps = float('nan')
try:
    _mu_c = _pred_full[_valid].double()
    _sd_c = _pred_std[_valid].double().clamp_min(1e-6)
    _y_c  = _targets[_valid].double()
    _w_c  = (_y_c - _mu_c) / _sd_c
    _Phi  = 0.5 * (1.0 + torch.erf(_w_c / _m42.sqrt(2.0)))
    _phi  = torch.exp(-0.5 * _w_c * _w_c) / _m42.sqrt(2.0 * _m42.pi)
    _crps_pix = _sd_c * (_w_c * (2.0 * _Phi - 1.0) + 2.0 * _phi - 1.0 / _m42.sqrt(_m42.pi))
    _crps = float(_crps_pix.mean().item())
except Exception as _e:
    print(f'[warn] CRPS failed : {_e}')
_dm['crps_gaussian'] = _crps

# 3. Intensity histogram L1 distance (proxy LHD).
_lhd = float('nan')
try:
    _p_h = _pred_full[_valid].double().flatten()
    _t_h = _targets[_valid].double().flatten()
    _lo = float(torch.minimum(_p_h.min(), _t_h.min()).item())
    _hi = float(torch.maximum(_p_h.max(), _t_h.max()).item())
    if _hi > _lo:
        _hp = torch.histc(_p_h.float(), bins=100, min=_lo, max=_hi)
        _ht = torch.histc(_t_h.float(), bins=100, min=_lo, max=_hi)
        _hp = _hp / _hp.sum().clamp_min(1.0)
        _ht = _ht / _ht.sum().clamp_min(1.0)
        _lhd = float((_hp - _ht).abs().sum().item())
except Exception as _e:
    print(f'[warn] histogram distance failed : {_e}')
_dm['intensity_hist_distance_L1'] = _lhd

# Secondary refs (same as BS42).
_dm['rapsd_distance']         = float(_rapsd_d) if _rapsd_d is not None else None
_dm['rmse_secondary']         = float(_rmse)
_dm['mae_secondary']          = float(_mae)
_dm['spread_mean']            = float(_spread)
_dm['pearson_global_secondary'] = float(_corr_global) if _corr_global == _corr_global else None

print(f'  Spread-skill ratio     : {_ssr:.4f}   (target ~1.0)')
print(f'  CRPS (gaussian)        : {_crps:.6f}')
print(f'  Hist. distance (~LHD)  : {_lhd:.4f}')
print(f'  RAPSD distance         : {_dm["rapsd_distance"]}')

_dm_path = PHASE6_REF_DIR / 'domain_metrics.json'
_dm_path.write_text(_j42.dumps(_dm, indent=2), encoding='utf-8')
print(f'\n[Cell 7] Saved : {_dm_path}')


In [ ]:
# === Cell 8 : run_aligned_eval -- CDD/Rx1Day/R10/seasonal/PSD per GCM ===
# Streams the val_dataloader, builds full HR fields in log1p space, then calls
# the cGAN-aligned eval (vendored). Writes one JSON per GCM in PHASE6_REF_DIR.
from st_cdgm.evaluation.aligned_eval import run_aligned_eval
import numpy as _np44

EVAL_GCM_TAG = str(globals().get('EVAL_GCM_TAG', 'ACCESS-CM2'))
EVAL_IN_DIST = bool(globals().get('EVAL_IN_DIST', True))
_ALIGNED_K   = int(globals().get('ALIGNED_K_SAMPLES', min(K_SAMPLES, 16)))

print(f'[Cell 8] Aligned eval | GCM={EVAL_GCM_TAG} | in_dist={EVAL_IN_DIST} '
      f'| variant=dualpath | K={_ALIGNED_K}')

_pred_seq, _truth_seq, _time_seq = [], [], []
_n_skip_time = 0

with torch.no_grad():
    for _conv in iterate_batches(val_dataloader, builder, DEVICE):
        for _b in _conv:
            _cond, _muHR, _blog, _tgt = _build_inputs(_b)
            _ens = torch.stack([_sample_once(_cond, _muHR, _blog)
                                for _ in range(_ALIGNED_K)], dim=0).mean(dim=0)
            # Full HR log1p field = baseline_log + mu_HR + delta_hat.
            _full_log  = (_blog if _blog is not None else 0.0) + \
                         (_muHR if _muHR is not None else 0.0) + _ens
            _truth_log = (_blog if _blog is not None else 0.0) + \
                         (_muHR if _muHR is not None else 0.0) + _tgt
            for _i in range(_full_log.shape[0]):
                _pred_seq.append(_full_log[_i].squeeze().detach().float().cpu().numpy())
                _truth_seq.append(_truth_log[_i].squeeze().detach().float().cpu().numpy())
            _tt = _b.get('time', None)
            if _tt is not None:
                _arr = _np44.atleast_1d(_np44.asarray(_tt))
                _time_seq.append(_arr.ravel()[-1])
            else:
                _n_skip_time += 1

_T = len(_pred_seq)
print(f'[Cell 8] Collected T={_T} timesteps  (skip_time={_n_skip_time})')

if _T == 0:
    print('[Cell 8] No predictions collected. Skip aligned eval.')
else:
    if len(_time_seq) == _T:
        _times = _np44.asarray(_time_seq, dtype='datetime64[ns]')
    else:
        print(f'[Cell 8] WARN : {_n_skip_time} samples without "time" -> synthetic daily axis.')
        _times = _np44.arange(_T, dtype='datetime64[D]').astype('datetime64[ns]')
    _out_path = PHASE6_REF_DIR / f'aligned_metrics_{EVAL_GCM_TAG}_dualpath.json'
    _res = run_aligned_eval(
        pred_fields=_pred_seq, truth_fields=_truth_seq, times=_times,
        out_path=_out_path, gcm=EVAL_GCM_TAG, run_variant='dualpath',
        in_distribution=EVAL_IN_DIST, space='log1p', thresh=1.0, k_samples=_ALIGNED_K,
    )
    print(f'[Cell 8] indices bias : '
          + ', '.join(f'{k}={v:+.3f}' for k, v in _res['indices'].items() if k.endswith('_bias')))
    print(f'[Cell 8] PSD distance = {_res["psd_distance"]:.5f}')
    print(f'[Cell 8] Saved : {_out_path}')


In [ ]:
# === Cell 9 : COMPARAISON 3-WAY (causal | non-causal | phase6_dualpath) ===
# Replique EXACTEMENT la structure de st_cdgm_causal_vs_noncausal_comparison.ipynb
# (cell cmp_quant) etendue pour 3 modeles + metriques additionnelles.
import json
from pathlib import Path
from collections import Counter

def _load_json(p):
    p = Path(p)
    if not p.exists():
        print(f'[warn] absent : {p}')
        return {}
    try:
        return json.loads(p.read_text(encoding='utf-8'))
    except Exception as e:
        print(f'[warn] echec lecture {p}: {e}')
        return {}

def _flatten(d, prefix=''):
    out = {}
    if not isinstance(d, dict):
        return out
    for k, v in d.items():
        key = f'{prefix}{k}'
        if isinstance(v, dict):
            out.update(_flatten(v, key + '.'))
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            out[key] = float(v)
    return out

# --- Load JSONs : fv + dm + aligned per model ---
causal_fv    = _load_json(CAUSAL_REF_DIR    / 'final_validation_metrics.json')
causal_dm    = _load_json(CAUSAL_REF_DIR    / 'domain_metrics.json')
causal_al    = _load_json(CAUSAL_REF_DIR    / 'aligned_metrics_ACCESS-CM2_causal.json')

noncausal_fv = _load_json(NONCAUSAL_REF_DIR / 'final_validation_metrics.json')
noncausal_dm = _load_json(NONCAUSAL_REF_DIR / 'domain_metrics.json')
noncausal_al = _load_json(NONCAUSAL_REF_DIR / 'aligned_metrics_ACCESS-CM2_noncausal.json')

phase6_fv    = _load_json(PHASE6_REF_DIR    / 'final_validation_metrics.json')
phase6_dm    = _load_json(PHASE6_REF_DIR    / 'domain_metrics.json')
phase6_al    = _load_json(PHASE6_REF_DIR    / 'aligned_metrics_ACCESS-CM2_dualpath.json')

print('Cles chargees :')
print(f'  causal    fv={"OK" if causal_fv else "MISS"}  dm={"OK" if causal_dm else "MISS"}  aligned={"OK" if causal_al else "MISS"}')
print(f'  noncausal fv={"OK" if noncausal_fv else "MISS"}  dm={"OK" if noncausal_dm else "MISS"}  aligned={"OK" if noncausal_al else "MISS"}')
print(f'  phase6    fv={"OK" if phase6_fv else "MISS"}  dm={"OK" if phase6_dm else "MISS"}  aligned={"OK" if phase6_al else "MISS"}')

# Flatten each source separately (preserve fv/dm/aligned distinction comme l'original)
c_fv, c_dm, c_al = _flatten(causal_fv), _flatten(causal_dm), _flatten(causal_al)
n_fv, n_dm, n_al = _flatten(noncausal_fv), _flatten(noncausal_dm), _flatten(noncausal_al)
p_fv, p_dm, p_al = _flatten(phase6_fv), _flatten(phase6_dm), _flatten(phase6_al)

def _val(model, source, key):
    src_map = {
        'causal':    {'fv': c_fv, 'dm': c_dm, 'al': c_al},
        'noncausal': {'fv': n_fv, 'dm': n_dm, 'al': n_al},
        'phase6':    {'fv': p_fv, 'dm': p_dm, 'al': p_al},
    }
    return src_map[model][source].get(key, None)

# HEADLINE etendu : 16 metriques organisees par categorie
HEADLINE = [
    # --- Skill (final_validation_metrics.json) ---
    ('Pearson global',         'fv', 'pearson_corr.global',        'up'),
    ('Pearson par-ech.',       'fv', 'pearson_corr.per_sample_avg','up'),
    ('RMSE',                   'fv', 'rmse',                       'down'),
    ('MAE',                    'fv', 'mae',                        'down'),
    # --- Spread / spectral ---
    ('Spread (ens.)',          'fv', 'spread_mean',                'info'),
    ('RAPSD distance',         'fv', 'rapsd_distance',             'down'),
    ('Spread-skill ratio',     'dm', 'spread_skill_ratio',         'to1'),
    # --- Probabilistique ---
    ('CRPS (gaussien)',        'dm', 'crps_gaussian',              'down'),
    ('Hist. distance (~LHD)',  'dm', 'intensity_hist_distance_L1', 'down'),
    # --- Extremes (F1) ---
    ('F1 @ p95',               'fv', 'f1_extremes.p95',            'up'),
    ('F1 @ p99',               'fv', 'f1_extremes.p99',            'up'),
    # --- Causalite ---
    ('mu_HR abl. delta/sig',   'fv', 'mu_HR_ablation.delta_signal_ratio_avg', 'info'),
    # --- Climate indices alignes (sens 'abs_min' = minimal absolute bias) ---
    ('CDD bias (j)',           'al', 'indices.cdd_bias',           'abs_min'),
    ('Rx1day bias (mm)',       'al', 'indices.rx1day_bias',        'abs_min'),
    ('R10day bias (j)',        'al', 'indices.r10day_bias',        'abs_min'),
    ('PSD distance',           'al', 'psd_distance',               'down'),
]

def _winner3(cv, nv, pv, sense):
    vals = {'causal': cv, 'noncausal': nv, 'phase6': pv}
    valid = {k: v for k, v in vals.items() if v is not None}
    if not valid:
        return '-'
    if sense == 'up':
        return max(valid, key=valid.get)
    if sense == 'down':
        return min(valid, key=valid.get)
    if sense == 'to1':
        return min(valid, key=lambda k: abs(valid[k] - 1))
    if sense == 'abs_min':
        return min(valid, key=lambda k: abs(valid[k]))
    return '-'

rows = []
print()
print('=' * 105)
print('COMPARAISON 3-WAY  (causal = ST-CDGM | non-causal = CorrDiff vanilla | phase6 = Dual-Path)')
print('=' * 105)
print(f'{"Metrique":<24}{"causal":>13}{"non-causal":>13}{"phase6":>13}{"sens":>9}{"avantage":>15}')
print('-' * 105)
for label, source, key, sense in HEADLINE:
    cv = _val('causal', source, key)
    nv = _val('noncausal', source, key)
    pv = _val('phase6', source, key)
    win = _winner3(cv, nv, pv, sense)
    def _fmt(x):
        return f'{x:+.4f}' if isinstance(x, float) else '-'
    print(f'{label:<24}{_fmt(cv):>13}{_fmt(nv):>13}{_fmt(pv):>13}{sense:>9}{win:>15}')
    rows.append({
        'metric':      label,
        'source':      source,
        'key':         key,
        'sense':       sense,
        'causal':      cv,
        'noncausal':   nv,
        'phase6':      pv,
        'winner':      win,
    })
print('-' * 105)

# --- Bilan par modele ---
counts = Counter(r['winner'] for r in rows if r['winner'] != '-')
print()
print('Bilan (nb metriques gagnees, hors sens=info) :')
for variant, n in counts.most_common():
    print(f'  {variant:12s} : {n}')

# --- Deltas headline : phase6 vs causal / phase6 vs noncausal ---
def _delta(a, b):
    if a is None or b is None: return None
    return a - b

print()
print('Deltas headline (phase6 - reference) :')
print(f'  {"Metric":<24}{"phase6-causal":>15}{"phase6-noncausal":>18}')
for label, source, key, sense in HEADLINE:
    cv = _val('causal', source, key)
    nv = _val('noncausal', source, key)
    pv = _val('phase6', source, key)
    dc = _delta(pv, cv)
    dn = _delta(pv, nv)
    fmt = lambda x: f'{x:+.4f}' if isinstance(x, float) else '-'
    print(f'  {label:<24}{fmt(dc):>15}{fmt(dn):>18}')

# --- Sauvegarde JSON ---
try:
    out = PHASE6_REF_DIR / 'comparison_3way_table.json'
    out.write_text(json.dumps(rows, indent=2, default=str), encoding='utf-8')
    print()
    print(f'Table sauvegardee : {out}')
except Exception as e:
    print(f'[warn] sauvegarde echouee : {e}')

print()
print('=' * 105)
print('Note : un quasi-match in-distribution est ATTENDU. L\'apport causal/dualpath se mesure surtout en :')
print('  1. OOD (CMIP6 EC-Earth3, NorESM2-MM) - a faire dans une version complete')
print('  2. Interpretabilite (DAG auditable, Q_phys=0.99 preservee)')
print('  3. Extremes (F1@p99) et structure spatiale (RAPSD, PSD distance)')
print('=' * 105)
